In [13]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from scipy.optimize import least_squares

sys.path.insert(0, str(Path("../../nogse_pipeline/src")))
from models.model_fitting import M_ogse_mixed_offset, M_ogse_tort
from plotting.publication.tc_param_vars import _roi_color_map, DEFAULT_BRAIN_MARKERS

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

MASTER_PATH  = Path("../../analysis/brains/ogse_experiments/master.long.parquet")
OUT_DIR      = Path("../../analysis/brains/ogse_experiments/lab/fit_rest-tort_CC-a1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTIONS  = ["tra", "long"]
N_LIST      = [4, 8]
TD_LIST     = None
# ROI_LIST
#    None → All ROIS
#    ["AntCC","MidAntCC","CentralCC","MidPostCC","PostCC"] → CC
#    ["Left-Lateral-Ventricle", "Right-Lateral-Ventricle"] → Ventricles
ROI_LIST    = ["AntCC","MidAntCC","CentralCC","MidPostCC","PostCC"]
G_COLUMN    = "g_thorsten"
G_CORRECTION_COLUMN = "grad_correction_factor"
Y_COLUMN    = "value"   # "value" (raw a.u.) or "value_norm" (normalised by S0)
D0_FIXED    = 3.2e-12       # m²/ms, fixed free-water diffusivity

# ALPHA_GLOBAL or TC_GLOBAL: scope for a shared parameter fit
#   None         → per-curve (one value per td, N, direction)
#   "direction"  → one value per (subj, roi, direction), shared across td & N
#   "roi"        → one value per (subj, roi), shared across td, N & direction
#   "subj"       → one value per subject, shared across all roi, td, N & direction
#   "all"        → one value for all data

# ── Correlation time tc1 (restricted / mixed component) ──────────────────────
TC1_FIXED   = None           # None → fitted per curve; float → pinned [ms]; "master" → from master table
TC1_INIT    = 2.0            # ms, initial guess
TC1_BOUNDS  = (0.05, 1000.0) # (lower, upper) ms
TC1_GLOBAL  = "direction"    # None → per-curve; "direction"/"roi"/"subj"/"all" → shared scope
TC1_MASTER_COL = "tc_ms"

# ── Tortuosity factor α1 (restricted / mixed component) ──────────────────────
ALPHA1_FIXED  = 0         # None → fitted; float → pinned; "master" → from master table
ALPHA1_INIT   = 0.5
ALPHA1_BOUNDS = (0.0, 1.0)
ALPHA1_GLOBAL = "direction"
ALPHA1_MASTER_COL = "alpha_macro"

# ── Tortuosity factor α2 (tort component — no tc) ────────────────────────────
ALPHA2_FIXED  = None
ALPHA2_INIT   = 0.5
ALPHA2_BOUNDS = (0.0, 1.0)
ALPHA2_GLOBAL = "direction"
ALPHA2_MASTER_COL = "alpha_macro"

# ── Mixing fractions f1, f2 ───────────────────────────────────────────────────
# Model:  S(G) = M0 · (f1·S_mixed(tc1,α1,G) + f2·S_tort(α2,G))
# where   M0 = sqrt(S0² − RN²)  (Rician correction)
#
# F_TOTAL : float → f1 + f2 is constrained to F_TOTAL (f2 = F_TOTAL − f1, not fitted);
#           None  → f1 and f2 are independently free.
# F1_FIXED: None  → f1 is a free parameter;
#           float → f1 is pinned to this value.
#
# F1_GLOBAL: how many curves share the same f1 when it is free (F1_FIXED = None). (analogous to TC1_GLOBAL / ALPHA1_GLOBAL)
#   None        → per-curve (one f1 per td, N, direction)
#   "direction" → one f1 per (subj, roi, direction), shared across td & N
#   "roi"       → one f1 per (subj, roi), shared across td, N & direction
#   "subj"      → one f1 per subject
#   "all"       → one f1 for all data
F_TOTAL   = 1.0    # typically 1.0; set None to let f1 + f2 float freely
F1_FIXED  = None   # None → fitted; float → pinned
F1_GLOBAL = None  # scope for shared f1

# ── Model name ────────────────────────────────────────────────────────────────
MODEL_NAME = 'rest-tort'

# ── Rician noise floor RN (signal units) ─────────────────────────────────────
# Signal model:  S(G) = sqrt( (M0·(f1·S_mixed + f2·S_tort))² + RN² )
#                where  M0 = sqrt(S0² − RN²)  [derived from data, not fitted]
#
#   None                        → no noise correction (RN = 0)
#   float                       → same RN for all subjects, td, and roi
#   {td: float}                 → fixed per td, same for all subjects and roi
#   {subj: float}               → fixed per subject, same for all td and roi
#   {subj: {td: float}}         → fixed per subject and td, same for all roi
#   {subj: {td: {roi: float}}}  → fixed per subject, td, and roi

RN_FIXED = {
    "BRAIN": {
        76.0:  {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
        90.0:  {"AntCC": 32, "MidAntCC": 38, "CentralCC": 36, "MidPostCC": 42, "PostCC": 36},
        120.0: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        143.4: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        210.0: {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
    },
    "LUDG": {
        90.0:  {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
        120.0: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        143.4: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        210.0: {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
    },
    "MBBL": {
        90.0:  {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
        120.0: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        143.4: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        210.0: {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
    },
}

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════

df = pd.read_parquet(MASTER_PATH)

data = df[
    (df.row_kind == "signal_rotated") &
    (df.direction.isin(DIRECTIONS)) &
    (df.N.isin(N_LIST)) &
    (df.stat == "avg")
].copy()

missing_cols = [c for c in [G_COLUMN, Y_COLUMN, G_CORRECTION_COLUMN] if c is not None and c not in data.columns]
if missing_cols:
    raise KeyError(f"Missing column(s): {missing_cols}")

data["G_fit"] = data[G_COLUMN]
if G_CORRECTION_COLUMN is not None:
    data["G_fit"] = data["G_fit"] * data[G_CORRECTION_COLUMN]
data["y_raw"] = data["value"]   # raw signal — always kept to derive S0
data["y_fit"] = data[Y_COLUMN]

G_LABEL = f"Modulation gradient G [mT/m] ({G_COLUMN})"
Y_LABEL = f"{Y_COLUMN} [a.u.]"

if TD_LIST is not None:
    data = data[data.td_ms.isin(TD_LIST)]
if ROI_LIST is not None:
    data = data[data.roi.isin(ROI_LIST)]

print(f"Rows loaded: {len(data)}")
print("Groups (subj, roi, dir):", data.groupby(["subj", "roi", "direction"]).ngroups)
print(f"Gradient column: {G_COLUMN}" + (f" × {G_CORRECTION_COLUMN}" if G_CORRECTION_COLUMN else ""))
print(f"Signal column:   {Y_COLUMN}")
data[["subj", "roi", "direction", "td_ms", "N", "G_fit", "y_fit"]]

Rows loaded: 2860
Groups (subj, roi, dir): 30
Gradient column: g_thorsten × grad_correction_factor
Signal column:   value


,subj,roi,direction,td_ms,N,G_fit,y_fit
5302,BRAIN,AntCC,tra,90.0,4,0.000000,250.168675
5303,BRAIN,AntCC,tra,90.0,4,6.213986,244.684694
5304,BRAIN,AntCC,tra,90.0,4,12.427971,228.825266
5305,BRAIN,AntCC,tra,90.0,4,18.641957,214.273122
5306,BRAIN,AntCC,tra,90.0,4,24.855943,194.742588
...,...,...,...,...,...,...,...
180395,MBBL,PostCC,long,210.0,8,35.446824,52.056472
180396,MBBL,PostCC,long,210.0,8,41.354627,50.485129
180397,MBBL,PostCC,long,210.0,8,47.262431,51.328731
180398,MBBL,PostCC,long,210.0,8,53.170235,52.411720


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT FUNCTIONS
#
# Signal model:  S(G) = sqrt( (M0·(f1·S_mixed + f2·S_tort))² + RN² )
#                where  M0 = sqrt(S0² − RN²)  [derived from data, not fitted]
#                S_mixed = M_ogse_mixed(tc1, α1, G)
#                S_tort  = M_ogse_tort(α2, G)   [no tc]
#                f1 + f2 = F_TOTAL  if F_TOTAL is not None, else f1, f2 are independent
# ══════════════════════════════════════════════════════════════════════════════

def _resolve_rn(subj, td, roi=None):
    """Return fixed RN for (subj, td, roi). Returns 0.0 when RN_FIXED is None."""
    if RN_FIXED is None:
        return 0.0
    if isinstance(RN_FIXED, dict):
        first_key = next(iter(RN_FIXED))
        if isinstance(first_key, str):               # {subj: ...}
            subj_entry = RN_FIXED.get(subj)
            if subj_entry is None:
                return 0.0
            if isinstance(subj_entry, dict):         # {subj: {td: ...}}
                td_entry = subj_entry.get(float(td), subj_entry.get(td))
                if td_entry is None:
                    return 0.0
                if isinstance(td_entry, dict):       # {subj: {td: {roi: float}}}
                    return float(td_entry.get(roi, 0.0))
                return float(td_entry)               # {subj: {td: float}}
            return float(subj_entry)                 # {subj: float}
        return float(RN_FIXED.get(float(td), RN_FIXED.get(td, 0.0)))  # {td: float}
    return float(RN_FIXED)


def _m0_from_s0(s0_raw, rn):
    """M0 = sqrt(S0² − RN²). For value_norm returns M0/S0."""
    m0 = np.sqrt(max(float(s0_raw)**2 - float(rn)**2, 0.0))
    if Y_COLUMN == "value_norm":
        return m0 / float(s0_raw) if float(s0_raw) > 0 else 1.0
    return m0


def _get_master_value(subj, roi, direction, col):
    """Look up a fixed value from the master table for one (subj, roi, direction) group."""
    rows = df[(df.subj == subj) & (df.roi == roi) & (df.direction == direction)]
    if col in rows.columns and not rows[col].isna().all():
        return float(rows[col].dropna().iloc[0])
    raise KeyError(f"'{col}' not found or all-NaN for {subj}/{roi}/{direction}")


def _resolve_tc1(subj, roi, direction):
    """Return fixed tc1 [ms] or None (→ to be fitted)."""
    if TC1_FIXED is None:
        return None
    if TC1_FIXED == "master":
        return _get_master_value(subj, roi, direction, TC1_MASTER_COL)
    return float(TC1_FIXED)


def _resolve_alpha1(subj, roi, direction):
    """Return fixed α1 or None (→ to be fitted)."""
    if ALPHA1_FIXED is None:
        return None
    if ALPHA1_FIXED == "master":
        return _get_master_value(subj, roi, direction, ALPHA1_MASTER_COL)
    return float(ALPHA1_FIXED)


def _resolve_alpha2(subj, roi, direction):
    """Return fixed α2 or None (→ to be fitted)."""
    if ALPHA2_FIXED is None:
        return None
    if ALPHA2_FIXED == "master":
        return _get_master_value(subj, roi, direction, ALPHA2_MASTER_COL)
    return float(ALPHA2_FIXED)


def _predict(td, G, N, tc1, alpha1, alpha2, f1, f2, M0, rn):
    """S(G) = sqrt( (M0·(f1·S_mixed + f2·S_tort))² + RN² )"""
    with np.errstate(over="ignore", invalid="ignore"):
        s1  = M_ogse_mixed_offset(td, G, N, td / N, tc1, alpha1, 1, D0_FIXED, 0, 0)
        s2  = M_ogse_tort(td, G, N, td / N, alpha2, 1, D0_FIXED)
        sig = M0 * (f1 * s1 + f2 * s2)
        return np.sqrt(sig**2 + rn**2) if rn != 0.0 else sig


def _param_stderr(result, param_idx):
    """1-σ error on params[param_idx] from the least-squares Jacobian."""
    dof = result.fun.size - result.x.size
    if dof <= 0 or result.x.size == 0 or param_idx >= result.x.size:
        return np.nan
    try:
        cov = np.linalg.pinv(result.jac.T @ result.jac) * (2.0 * result.cost / dof)
        return float(np.sqrt(max(float(cov[param_idx, param_idx]), 0.0)))
    except np.linalg.LinAlgError:
        return np.nan


def fit_curve(td, N, G, y, subj=None, roi=None, direction=None, s0=None):
    """Fit tc1, α1, α2, f1 for ONE (td, N, direction) curve.

    Returns (tc1, alpha1, alpha2, f1, f2, result).
    tc1 is optimised in log-space for numerical stability.
    f1 is bounded in [0, F_TOTAL] when F_TOTAL is not None, else [0, 1].
    When F_TOTAL is not None: f2 = F_TOTAL − f1 (not a free parameter).
    When F_TOTAL is None: f2 is fitted independently in [0, 1].
    """
    rn = _resolve_rn(subj, td, roi)
    M0 = _m0_from_s0(s0, rn) if s0 is not None else float(y.max())

    tc1_fv  = _resolve_tc1(subj, roi, direction)
    al1_fv  = _resolve_alpha1(subj, roi, direction)
    al2_fv  = _resolve_alpha2(subj, roi, direction)
    f1_fv   = float(F1_FIXED) if F1_FIXED is not None else None
    f_total = float(F_TOTAL)  if F_TOTAL  is not None else None

    free, x0, lo, hi = [], [], [], []
    if tc1_fv is None:
        free.append("tc1")
        x0.append(np.log(TC1_INIT)); lo.append(np.log(TC1_BOUNDS[0])); hi.append(np.log(TC1_BOUNDS[1]))
    if al1_fv is None:
        free.append("alpha1")
        x0.append(ALPHA1_INIT); lo.append(ALPHA1_BOUNDS[0]); hi.append(ALPHA1_BOUNDS[1])
    if al2_fv is None:
        free.append("alpha2")
        x0.append(ALPHA2_INIT); lo.append(ALPHA2_BOUNDS[0]); hi.append(ALPHA2_BOUNDS[1])
    if f1_fv is None:
        free.append("f1")
        f1_hi = f_total if f_total is not None else 1.0
        x0.append(0.5 * f1_hi); lo.append(0.0); hi.append(f1_hi)
    if f_total is None:  # f2 is also free (independent of f1)
        free.append("f2")
        f2_init = 1.0 - (f1_fv if f1_fv is not None else 0.5)
        x0.append(max(0.0, f2_init)); lo.append(0.0); hi.append(1.0)

    def _unpack(params):
        idx = 0
        if "tc1" in free:
            tc1 = np.exp(params[idx]); idx += 1
        else:
            tc1 = tc1_fv
        if "alpha1" in free:
            alpha1 = params[idx]; idx += 1
        else:
            alpha1 = al1_fv
        if "alpha2" in free:
            alpha2 = params[idx]; idx += 1
        else:
            alpha2 = al2_fv
        if "f1" in free:
            f1 = params[idx]; idx += 1
        else:
            f1 = f1_fv
        if f_total is not None:
            f2 = f_total - f1
        elif "f2" in free:
            f2 = params[idx]
        else:
            f2 = 1.0 - f1
        return tc1, alpha1, alpha2, f1, f2

    if not free:
        tc1, alpha1, alpha2, f1, f2 = _unpack([])
        fun = _predict(td, G, N, tc1, alpha1, alpha2, f1, f2, M0, rn) - y
        mock = type("FixedResult", (), {
            "x": np.array([]), "fun": fun, "jac": np.zeros((len(fun), 0)),
            "cost": 0.5 * float(np.sum(fun**2)), "success": True,
        })()
        return tc1, alpha1, alpha2, f1, f2, mock

    def residuals(params):
        tc1, alpha1, alpha2, f1, f2 = _unpack(params)
        out = _predict(td, G, N, tc1, alpha1, alpha2, f1, f2, M0, rn) - y
        return np.where(np.isfinite(out), out, 1e6)

    result = least_squares(residuals, x0, bounds=(lo, hi), method="trf", max_nfev=10000)
    tc1, alpha1, alpha2, f1, f2 = _unpack(result.x)
    return tc1, alpha1, alpha2, f1, f2, result


# ── Scope definitions for global fitting ─────────────────────────────────────
_SCOPE_RANK = {"all": 1, "subj": 2, "roi": 3, "direction": 4}
_SCOPE_COLS = {
    "all":       [],
    "subj":      ["subj"],
    "roi":       ["subj", "roi"],
    "direction": ["subj", "roi", "direction"],
}


def _group_scope():
    """Return (scope_name, group_cols) for the finest scope across all global params."""
    candidates = [TC1_GLOBAL, ALPHA1_GLOBAL, ALPHA2_GLOBAL, F1_GLOBAL]
    ranks = [_SCOPE_RANK.get(s, 0) for s in candidates]
    finest = candidates[ranks.index(max(ranks))]
    return finest, _SCOPE_COLS.get(finest, [])


def fit_group_global(group_df):
    """Joint fit across all curves in group_df.

    Parameters shared within the group (when their _GLOBAL scope is not None):
      tc1, alpha1, alpha2, f1
    Parameters fitted independently per curve when their _GLOBAL is None.
    Parameters fixed (not optimised) when their _FIXED value is not None.

    Returns: (result, curves,
              per_tc1s, per_tc1_errs,
              per_al1s, per_al1_errs, per_al2s, per_al2_errs,
              per_f1s, per_f1_errs, per_f2s, M0s)
    tc errors are in ms (converted from log-space Jacobian: σ_tc = tc · σ_{log_tc}).
    """
    f_total = float(F_TOTAL) if F_TOTAL is not None else None
    f1_fv   = float(F1_FIXED) if F1_FIXED is not None else None

    curves = []
    for (subj, roi, direction), dir_grp in group_df.groupby(["subj", "roi", "direction"]):
        for td in sorted(float(t) for t in dir_grp.td_ms.unique()):
            sub = dir_grp[np.isclose(dir_grp.td_ms.astype(float), td)]
            for N in N_LIST:
                s = sub[sub.N == N].sort_values("G_fit")
                if s.empty:
                    continue
                s0_raw = float(s.loc[s.G_fit.abs() == s.G_fit.abs().min(), "y_raw"].iloc[0])
                rn = _resolve_rn(subj, td, roi)
                curves.append(dict(
                    subj=subj, roi=roi, direction=direction,
                    td=td, N=int(N), S0=s0_raw,
                    G=s.G_fit.values, y=s.y_fit.values,
                    M0=_m0_from_s0(s0_raw, rn),
                    rn=rn,
                    tc1_fv=_resolve_tc1(subj, roi, direction),
                    al1_fv=_resolve_alpha1(subj, roi, direction),
                    al2_fv=_resolve_alpha2(subj, roi, direction),
                ))

    M0s = {(c["subj"], c["roi"], c["direction"], c["td"], c["N"]): c["M0"] for c in curves}

    share_tc1    = (TC1_FIXED    is None and TC1_GLOBAL    is not None)
    share_alpha1 = (ALPHA1_FIXED is None and ALPHA1_GLOBAL is not None)
    share_alpha2 = (ALPHA2_FIXED is None and ALPHA2_GLOBAL is not None)
    share_f1     = (F1_FIXED     is None and F1_GLOBAL     is not None)
    percurve_tc1    = (TC1_FIXED    is None and TC1_GLOBAL    is None)
    percurve_alpha1 = (ALPHA1_FIXED is None and ALPHA1_GLOBAL is None)
    percurve_alpha2 = (ALPHA2_FIXED is None and ALPHA2_GLOBAL is None)
    percurve_f1     = (F1_FIXED     is None and F1_GLOBAL     is None)

    x0, lo, hi = [], [], []
    tc1_si = al1_si = al2_si = f1_si = None  # shared indices

    if share_tc1:
        tc1_si = len(x0)
        x0.append(np.log(TC1_INIT)); lo.append(np.log(TC1_BOUNDS[0])); hi.append(np.log(TC1_BOUNDS[1]))
    if share_alpha1:
        al1_si = len(x0)
        x0.append(ALPHA1_INIT); lo.append(ALPHA1_BOUNDS[0]); hi.append(ALPHA1_BOUNDS[1])
    if share_alpha2:
        al2_si = len(x0)
        x0.append(ALPHA2_INIT); lo.append(ALPHA2_BOUNDS[0]); hi.append(ALPHA2_BOUNDS[1])
    if share_f1:
        f1_si = len(x0)
        f1_hi = f_total if f_total is not None else 1.0
        x0.append(0.5 * f1_hi); lo.append(0.0); hi.append(f1_hi)

    tc1_ci = {}; al1_ci = {}; al2_ci = {}; f1_ci = {}; f2_ci = {}

    for i, c in enumerate(curves):
        if percurve_tc1:
            tc1_ci[i] = len(x0)
            x0.append(np.log(TC1_INIT)); lo.append(np.log(TC1_BOUNDS[0])); hi.append(np.log(TC1_BOUNDS[1]))
        if percurve_alpha1:
            al1_ci[i] = len(x0)
            x0.append(ALPHA1_INIT); lo.append(ALPHA1_BOUNDS[0]); hi.append(ALPHA1_BOUNDS[1])
        if percurve_alpha2:
            al2_ci[i] = len(x0)
            x0.append(ALPHA2_INIT); lo.append(ALPHA2_BOUNDS[0]); hi.append(ALPHA2_BOUNDS[1])
        if percurve_f1:
            f1_ci[i] = len(x0)
            f1_hi = f_total if f_total is not None else 1.0
            x0.append(0.5 * f1_hi); lo.append(0.0); hi.append(f1_hi)
        if f_total is None:  # f2 always needs its own per-curve param when F_TOTAL is None
            f2_ci[i] = len(x0)
            f2_init = 1.0 - (f1_fv if f1_fv is not None else 0.5)
            x0.append(max(0.0, f2_init)); lo.append(0.0); hi.append(1.0)

    def _curve_params(params, i, c):
        if share_tc1:      tc1 = np.exp(params[tc1_si])
        elif percurve_tc1: tc1 = np.exp(params[tc1_ci[i]])
        else:              tc1 = c["tc1_fv"]

        if share_alpha1:      alpha1 = params[al1_si]
        elif percurve_alpha1: alpha1 = params[al1_ci[i]]
        else:                 alpha1 = c["al1_fv"]

        if share_alpha2:      alpha2 = params[al2_si]
        elif percurve_alpha2: alpha2 = params[al2_ci[i]]
        else:                 alpha2 = c["al2_fv"]

        if share_f1:      f1 = params[f1_si]
        elif percurve_f1: f1 = params[f1_ci[i]]
        else:             f1 = f1_fv

        if f_total is not None:
            f2 = f_total - f1
        elif i in f2_ci:
            f2 = params[f2_ci[i]]
        else:
            f2 = 1.0 - f1

        return tc1, alpha1, alpha2, f1, f2

    if not x0:
        f1_val = f1_fv if f1_fv is not None else 0.5
        f2_val = (f_total - f1_val) if f_total is not None else (1.0 - f1_val)
        fun = np.concatenate([
            _predict(c["td"], c["G"], c["N"],
                     c["tc1_fv"], c["al1_fv"], c["al2_fv"],
                     f1_val, f2_val, c["M0"], c["rn"]) - c["y"]
            for c in curves
        ])
        result = type("FixedResult", (), {
            "x": np.array([]), "fun": fun, "jac": np.zeros((len(fun), 0)),
            "cost": 0.5 * float(np.sum(fun**2)), "success": True,
        })()
        return (result, curves,
                [c["tc1_fv"] for c in curves], [np.nan] * len(curves),
                [c["al1_fv"] for c in curves], [np.nan] * len(curves),
                [c["al2_fv"] for c in curves], [np.nan] * len(curves),
                [f1_val] * len(curves), [np.nan] * len(curves),
                [f2_val] * len(curves), M0s)

    def residuals(params):
        res = []
        for i, c in enumerate(curves):
            tc1, alpha1, alpha2, f1, f2 = _curve_params(params, i, c)
            out = _predict(c["td"], c["G"], c["N"], tc1, alpha1, alpha2, f1, f2, c["M0"], c["rn"]) - c["y"]
            res.append(np.where(np.isfinite(out), out, 1e6))
        return np.concatenate(res)

    result = least_squares(residuals, x0, bounds=(lo, hi), method="trf", max_nfev=10000)

    per_tc1s, per_tc1e = [], []
    per_al1s, per_al1e = [], []
    per_al2s, per_al2e = [], []
    per_f1s,  per_f1e  = [], []
    per_f2s            = []

    for i, c in enumerate(curves):
        tc1, alpha1, alpha2, f1, f2 = _curve_params(result.x, i, c)
        per_tc1s.append(tc1)
        per_al1s.append(alpha1); per_al2s.append(alpha2)
        per_f1s.append(f1);      per_f2s.append(f2)
        per_tc1e.append(tc1 * _param_stderr(result, tc1_ci[i]) if percurve_tc1 else
                        tc1 * _param_stderr(result, tc1_si)    if share_tc1    else np.nan)
        per_al1e.append(_param_stderr(result, al1_ci[i]) if percurve_alpha1 else
                        _param_stderr(result, al1_si)    if share_alpha1    else np.nan)
        per_al2e.append(_param_stderr(result, al2_ci[i]) if percurve_alpha2 else
                        _param_stderr(result, al2_si)    if share_alpha2    else np.nan)
        per_f1e.append(_param_stderr(result, f1_ci[i]) if percurve_f1 else
                       _param_stderr(result, f1_si)    if share_f1    else np.nan)

    return (result, curves,
            per_tc1s, per_tc1e,
            per_al1s, per_al1e, per_al2s, per_al2e,
            per_f1s, per_f1e, per_f2s, M0s)

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT LOOP  —  runs all (subj, roi, dir, td, N) groups
# ══════════════════════════════════════════════════════════════════════════════

rows      = []
fit_store = {}

# Per-curve param indices in result.x (for error extraction when no global scope is used)
_idx = 0
_tc1_pi  = _idx if TC1_FIXED    is None else None; _idx += (TC1_FIXED    is None)
_al1_pi  = _idx if ALPHA1_FIXED is None else None; _idx += (ALPHA1_FIXED is None)
_al2_pi  = _idx if ALPHA2_FIXED is None else None; _idx += (ALPHA2_FIXED is None)
_f1_pi   = _idx if F1_FIXED     is None else None


def _tc_err(result, tc, pidx):
    return tc * _param_stderr(result, pidx) if pidx is not None else np.nan

def _scalar_err(result, pidx):
    return _param_stderr(result, pidx) if pidx is not None else np.nan


_any_global = any(v is not None for v in [TC1_GLOBAL, ALPHA1_GLOBAL, ALPHA2_GLOBAL, F1_GLOBAL])

if not _any_global:
    # ── Per-curve fit: one (tc1, α1, α2, f1) per (subj, roi, dir, td, N) ──
    for (subj, roi, direction), grp in data.groupby(["subj", "roi", "direction"]):
        tds = sorted(float(t) for t in grp.td_ms.unique())
        fit_store[(subj, roi, direction)] = {"tds": tds, "fits": {td: {} for td in tds}}

        for td in tds:
            sub = grp[np.isclose(grp.td_ms.astype(float), td)]
            for N in N_LIST:
                s = sub[sub.N == N].sort_values("G_fit")
                if s.empty:
                    continue
                G, y = s.G_fit.values, s.y_fit.values
                s0_raw = float(s.loc[s.G_fit.abs() == s.G_fit.abs().min(), "y_raw"].iloc[0])
                tc1, alpha1, alpha2, f1, f2, result = fit_curve(
                    td, N, G, y, subj=subj, roi=roi, direction=direction, s0=s0_raw
                )
                rn = _resolve_rn(subj, td, roi)
                M0 = _m0_from_s0(s0_raw, rn)

                print(
                    f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  N={N}  "
                    f"tc1={tc1:.3f} ms  α1={alpha1:.4f}  α2={alpha2:.4f}  "
                    f"f1={f1:.3f}  M0={M0:.4f}  RN={rn:.2f}  cost={result.cost:.3e}"
                )
                fit_store[(subj, roi, direction)]["fits"][td][N] = dict(
                    G=G, y=y, tc1=tc1, alpha1=alpha1, alpha2=alpha2,
                    f1=f1, f2=f2, M0=M0, RN=rn, S0=s0_raw,
                )
                rows.append(dict(
                    subj=subj, roi=roi, direction=direction, td_ms=td, N=N,
                    tc1_ms=tc1, tc1_err=_tc_err(result, tc1, _tc1_pi),
                    alpha1=alpha1, alpha1_err=_scalar_err(result, _al1_pi),
                    alpha2=alpha2, alpha2_err=_scalar_err(result, _al2_pi),
                    f1=f1, f1_err=_scalar_err(result, _f1_pi), f2=f2,
                    M0=M0, D0_m2ms=D0_FIXED, RN=rn,
                    model_name=MODEL_NAME,
                    g_column=G_COLUMN, g_correction_column=G_CORRECTION_COLUMN, y_column=Y_COLUMN,
                    cost=float(result.cost), success=bool(result.success),
                ))

else:
    # ── Global fit: shared params over the specified scope ────────────────────
    scope_name, scope_cols = _group_scope()
    groups = data.groupby(scope_cols) if scope_cols else [("all", data)]

    for group_key, group_df in groups:
        (result, curves,
         per_tc1s, per_tc1e,
         per_al1s, per_al1e, per_al2s, per_al2e,
         per_f1s, per_f1e, per_f2s, M0s) = fit_group_global(group_df)

        key_str = "  ".join(str(k) for k in group_key) if isinstance(group_key, tuple) else str(group_key)
        print(f"[{key_str}]  cost={result.cost:.3e}")

        for (subj, roi, direction), sub_grp in group_df.groupby(["subj", "roi", "direction"]):
            if (subj, roi, direction) not in fit_store:
                tds_srd = sorted(float(t) for t in sub_grp.td_ms.unique())
                fit_store[(subj, roi, direction)] = {"tds": tds_srd, "fits": {t: {} for t in tds_srd}}

        for c, tc1, tc1e, al1, al1e, al2, al2e, f1, f1e, f2 in zip(
            curves,
            per_tc1s, per_tc1e,
            per_al1s, per_al1e, per_al2s, per_al2e,
            per_f1s, per_f1e, per_f2s,
        ):
            subj, roi, direction, td, N = c["subj"], c["roi"], c["direction"], c["td"], c["N"]
            M0 = M0s[(subj, roi, direction, td, N)]
            fit_store[(subj, roi, direction)]["fits"][td][N] = dict(
                G=c["G"], y=c["y"], tc1=tc1, alpha1=al1, alpha2=al2,
                f1=f1, f2=f2, M0=M0, RN=c["rn"], S0=c["S0"],
            )
            rows.append(dict(
                subj=subj, roi=roi, direction=direction, td_ms=td, N=N,
                tc1_ms=tc1, tc1_err=tc1e,
                alpha1=al1, alpha1_err=al1e, alpha2=al2, alpha2_err=al2e,
                f1=f1, f1_err=f1e, f2=f2,
                M0=M0, D0_m2ms=D0_FIXED, RN=c["rn"],
                model_name=MODEL_NAME,
                g_column=G_COLUMN, g_correction_column=G_CORRECTION_COLUMN, y_column=Y_COLUMN,
                cost=float(result.cost), success=bool(result.success),
            ))

[BRAIN  AntCC  long]  cost=9.518e+02
[BRAIN  AntCC  tra]  cost=4.177e+02
[BRAIN  CentralCC  long]  cost=1.041e+03
[BRAIN  CentralCC  tra]  cost=7.657e+02
[BRAIN  MidAntCC  long]  cost=7.131e+02
[BRAIN  MidAntCC  tra]  cost=3.621e+02
[BRAIN  MidPostCC  long]  cost=1.996e+03
[BRAIN  MidPostCC  tra]  cost=1.800e+03
[BRAIN  PostCC  long]  cost=5.964e+02
[BRAIN  PostCC  tra]  cost=3.178e+02
[LUDG  AntCC  long]  cost=6.382e+02
[LUDG  AntCC  tra]  cost=5.481e+02
[LUDG  CentralCC  long]  cost=7.305e+02
[LUDG  CentralCC  tra]  cost=5.319e+02
[LUDG  MidAntCC  long]  cost=2.678e+03
[LUDG  MidAntCC  tra]  cost=7.044e+02
[LUDG  MidPostCC  long]  cost=2.349e+03
[LUDG  MidPostCC  tra]  cost=1.004e+03
[LUDG  PostCC  long]  cost=7.429e+02
[LUDG  PostCC  tra]  cost=4.960e+02
[MBBL  AntCC  long]  cost=3.824e+02
[MBBL  AntCC  tra]  cost=3.744e+02
[MBBL  CentralCC  long]  cost=8.237e+02
[MBBL  CentralCC  tra]  cost=6.068e+02
[MBBL  MidAntCC  long]  cost=5.709e+02
[MBBL  MidAntCC  tra]  cost=3.237e+02
[MBBL

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE RESULTS TABLE
# Columns: subj, roi, direction, td_ms, N,
#          tc1_ms, tc1_err,
#          alpha1, alpha1_err, alpha2, alpha2_err,
#          f1, f1_err, f2, M0, D0_m2ms, RN, cost, ...
# ══════════════════════════════════════════════════════════════════════════════

results_df = pd.DataFrame(rows)
results_df.to_excel(OUT_DIR / "fit_results.xlsx", index=False)
print(f"Saved {len(results_df)} rows → {OUT_DIR / 'fit_results.xlsx'}")
results_df

Saved 260 rows → ../../analysis/brains/ogse_experiments/lab/fit_rest-tort_CC-a1/fit_results.xlsx


,subj,roi,direction,td_ms,N,tc1_ms,tc1_err,alpha1,alpha1_err,alpha2,...,f2,M0,D0_m2ms,RN,model_name,g_column,g_correction_column,y_column,cost,success
0,BRAIN,AntCC,long,76.0,4,6.314461,3.470083,0.0,NaN,0.676830,...,0.262234,245.487080,3.200000e-12,37.0,rest-tort,g_thorsten,grad_correction_factor,value,951.793378,True
1,BRAIN,AntCC,long,76.0,8,6.314461,3.470083,0.0,NaN,0.676830,...,1.000000,276.616751,3.200000e-12,37.0,rest-tort,g_thorsten,grad_correction_factor,value,951.793378,True
2,BRAIN,AntCC,long,90.0,4,6.314461,3.470083,0.0,NaN,0.676830,...,0.861996,248.113615,3.200000e-12,32.0,rest-tort,g_thorsten,grad_correction_factor,value,951.793378,True
3,BRAIN,AntCC,long,90.0,8,6.314461,3.470083,0.0,NaN,0.676830,...,1.000000,245.347918,3.200000e-12,32.0,rest-tort,g_thorsten,grad_correction_factor,value,951.793378,True
4,BRAIN,AntCC,long,120.0,4,6.314461,3.470083,0.0,NaN,0.676830,...,0.707709,183.129514,3.200000e-12,36.0,rest-tort,g_thorsten,grad_correction_factor,value,951.793378,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,MBBL,PostCC,tra,120.0,8,2.066593,0.226886,0.0,NaN,0.931754,...,0.447448,244.573024,3.200000e-12,44.0,rest-tort,g_thorsten,grad_correction_factor,value,355.526124,True
256,MBBL,PostCC,tra,143.4,4,2.066593,0.226886,0.0,NaN,0.931754,...,0.628622,210.004099,3.200000e-12,44.0,rest-tort,g_thorsten,grad_correction_factor,value,355.526124,True
257,MBBL,PostCC,tra,143.4,8,2.066593,0.226886,0.0,NaN,0.931754,...,0.441949,187.101132,3.200000e-12,44.0,rest-tort,g_thorsten,grad_correction_factor,value,355.526124,True
258,MBBL,PostCC,tra,210.0,4,2.066593,0.226886,0.0,NaN,0.931754,...,0.754560,147.858041,3.200000e-12,51.0,rest-tort,g_thorsten,grad_correction_factor,value,355.526124,True


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT FITS — raw signal with Rician noise correction
# One figure per (subj, roi, dir): data points + fitted S(G) per (td, N)
# Top row:    S(G) = sqrt( (M0·(f1·S_mixed + f2·S_tort))² + RN² )
# Bottom row: signal contrast  ΔS = S(N_hi) − S(N_lo)  vs G
# ══════════════════════════════════════════════════════════════════════════════

G_plot = np.linspace(0, float(data.G_fit.max()), 300)
N_hi, N_lo = N_LIST[-1], N_LIST[0]

for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes_grid = plt.subplots(2, n_tds, figsize=(4 * n_tds, 7), sharey="row", squeeze=False)
    axes      = axes_grid[0]
    diff_axes = axes_grid[1]

    for ax, dax, td in zip(axes, diff_axes, tds):
        fits_td = store["fits"][td]
        title_lines = [f"td = {td:.1f} ms"]
        y_hat_per_N = {}
        for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
            fit = fits_td.get(N)
            if fit is None:
                continue
            ax.scatter(fit["G"], fit["y"], color=color, s=20, zorder=3, label=f"N={N} data")
            y_hat = _predict(td, G_plot, N,
                             fit["tc1"], fit["alpha1"], fit["alpha2"],
                             fit["f1"], fit["f2"], fit["M0"], fit["RN"])
            y_hat_per_N[N] = y_hat
            ax.plot(G_plot, y_hat, color=color, label=f"N={N} fit")
            title_lines.append(
                f"tc1={fit['tc1']:.2f} ms  α1={fit['alpha1']:.3f}  α2={fit['alpha2']:.3f}  "
                f"f1={fit['f1']:.3f}  RN={fit['RN']:.1f}  (N={N})"
            )
        ax.set_title("\n".join(title_lines), fontsize=6)
        ax.set_xlabel(G_LABEL)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.set_axisbelow(True)
        if N_hi in y_hat_per_N and N_lo in y_hat_per_N:
            dax.plot(G_plot, y_hat_per_N[N_hi] - y_hat_per_N[N_lo], color="k")
            dax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
        dax.set_title(f"td = {td:.1f} ms  (N={N_hi} − N={N_lo})", fontsize=8)
        dax.set_xlabel(G_LABEL)
        dax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        dax.set_axisbelow(True)

    axes[0].set_ylabel(Y_LABEL)
    diff_axes[0].set_ylabel(f"ΔS  (N={N_hi} − N={N_lo})")
    axes[0].legend(fontsize=6)
    fig.suptitle(f"{subj}  |  {roi}  |  {direction}", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Fit plots saved.")

Fit plots saved.


In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT parameters vs td  —  all subjects, grouped by ROI and direction
# One figure per (param, direction); one panel per ROI
# Colour = subject (lighter → darker within ROI palette), linestyle = N
# ══════════════════════════════════════════════════════════════════════════════

rois     = sorted(results_df.roi.unique())
subjects = sorted(results_df.subj.unique())
roi_colors   = _roi_color_map(rois)
_N_ls        = ["-", "--", "-.", ":"]
N_linestyles = {N: ls for N, ls in zip(N_LIST, _N_ls)}

for direction in DIRECTIONS:
    sub = results_df[results_df.direction == direction]
    for param, ylabel in [
        ("tc1_ms", "tc1 [ms]"),
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(1, len(rois), figsize=(4 * len(rois), 4), sharey=True)
        axes = np.atleast_1d(axes)

        for ax, roi in zip(axes, rois):
            base_color = roi_colors[roi]
            cmap = mcolors.LinearSegmentedColormap.from_list("", ["#cccccc", base_color])
            for i, subj in enumerate(subjects):
                shade  = cmap(0.3 + 0.7 * i / max(1, len(subjects) - 1))
                marker = DEFAULT_BRAIN_MARKERS[i % len(DEFAULT_BRAIN_MARKERS)]
                for N in N_LIST:
                    g = (
                        sub[(sub.roi == roi) & (sub.subj == subj) & (sub.N == N)]
                        .drop_duplicates("td_ms").sort_values("td_ms")
                    )
                    if g.empty:
                        continue
                    label = subj if N == N_LIST[0] else None
                    ax.scatter(g.td_ms, g[param], color=shade, marker=marker, s=60, zorder=3, label=label)
                    ax.plot(g.td_ms, g[param], color=shade, linewidth=0.8,
                            linestyle=N_linestyles[N])
            ax.set_title(roi, fontsize=9)
            ax.set_xlabel("td [ms]")
            ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)

        axes[0].set_ylabel(ylabel)
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, fontsize=7, loc="upper right", ncol=1)
        n_handles = [
            mlines.Line2D([], [], color="gray", linestyle=N_linestyles[N], label=f"N={N}")
            for N in N_LIST
        ]
        axes[-1].legend(handles=n_handles, fontsize=7, loc="lower right")
        fig.suptitle(f"{param} vs td — direction: {direction}", fontsize=10)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{direction}.png", dpi=120)
        plt.close(fig)

In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT NORMALIZED FITS  —  M0·(f1·S_mixed + f2·S_tort) with no Rician correction
# Useful to compare model shape without amplitude effects.
# Data is corrected: sqrt(|y² − RN²|) / M0 → compared to f1·S_mixed + f2·S_tort
# ══════════════════════════════════════════════════════════════════════════════

for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes_grid = plt.subplots(2, n_tds, figsize=(4 * n_tds, 7), sharey="row", squeeze=False)
    axes      = axes_grid[0]
    diff_axes = axes_grid[1]

    for ax, dax, td in zip(axes, diff_axes, tds):
        fits_td = store["fits"][td]
        y_hat_per_N = {}
        for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
            fit = fits_td.get(N)
            if fit is None:
                continue
            M0 = fit["M0"] if fit["M0"] != 0.0 else 1.0
            RN = fit["RN"]
            y_corr = np.sqrt(np.abs(fit["y"]**2 - RN**2)) / M0
            ax.scatter(fit["G"], y_corr, color=color, s=20, zorder=3, label=f"N={N} data corr")
            with np.errstate(over="ignore", invalid="ignore"):
                s1 = M_ogse_mixed_offset(td, G_plot, N, td / N, fit["tc1"], fit["alpha1"], 1.0, D0_FIXED, 0, 0)
                s2 = M_ogse_tort(td, G_plot, N, td / N, fit["alpha2"], 1.0, D0_FIXED)
                sig_norm = fit["f1"] * s1 + fit["f2"] * s2
            y_hat_per_N[N] = sig_norm
            ax.plot(G_plot, sig_norm, color=color, label=f"N={N} model")
        ax.set_title(f"td = {td:.1f} ms", fontsize=8)
        ax.set_xlabel(G_LABEL)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.set_axisbelow(True)
        if N_hi in y_hat_per_N and N_lo in y_hat_per_N:
            dax.plot(G_plot, y_hat_per_N[N_hi] - y_hat_per_N[N_lo], color="k")
            dax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
        dax.set_title(f"td = {td:.1f} ms  (N={N_hi} − N={N_lo})", fontsize=8)
        dax.set_xlabel(G_LABEL)
        dax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        dax.set_axisbelow(True)

    axes[0].set_ylabel(f"{Y_COLUMN} / M0 [a.u.]")
    diff_axes[0].set_ylabel(f"ΔS  (N={N_hi} − N={N_lo})")
    axes[0].legend(fontsize=6)
    fig.suptitle(f"{subj}  |  {roi}  |  {direction}  |  f1·S_mixed+f2·S_tort  (clean model)", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_norm_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Normalized model plots saved.")

Normalized model plots saved.


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT tc1, α1, α2, f1 and M0 vs td  —  per subject
# Rows = ROIs, columns = directions, colour/linestyle = N
# Shaded band = ±1σ from the Jacobian covariance
# ══════════════════════════════════════════════════════════════════════════════

subjects_plot = sorted(results_df.subj.unique())
rois_plot     = sorted(results_df.roi.unique())
Ns_plot       = sorted(results_df.N.unique())
dirs_plot     = [d for d in ["long", "tra"] if d in results_df.direction.unique()]
_N_ls_p       = ["-", "--", "-.", ":"]
N_ls          = {N: _N_ls_p[i % len(_N_ls_p)] for i, N in enumerate(Ns_plot)}

for subj in subjects_plot:
    for param, ylabel in [
        ("tc1_ms", "tc1 [ms]"),
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"), ("M0", "M0 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False,
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                s = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = s[s.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x    = g.td_ms.to_numpy(dtype=float)
                        yv   = g[param].to_numpy(dtype=float)
                        yerr = g[err_col].to_numpy(dtype=float)
                        ok   = np.isfinite(x) & np.isfinite(yv) & np.isfinite(yerr)
                        if np.any(ok):
                            ax.fill_between(x[ok], yv[ok] - yerr[ok], yv[ok] + yerr[ok],
                                            color=line.get_color(), alpha=0.18, linewidth=0)
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel}", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{subj}.png", dpi=120)
        plt.close(fig)

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT tc1, α1, α2, f1 and M0 vs td  —  per subject, log-scale y-axis
# Same layout as previous cell; useful to spot order-of-magnitude variation
# ══════════════════════════════════════════════════════════════════════════════

for subj in subjects_plot:
    for param, ylabel in [
        ("tc1_ms", "tc1 [ms]"),
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"), ("M0", "M0 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False,
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                s = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = s[s.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x     = g.td_ms.to_numpy(dtype=float)
                        yv    = g[param].to_numpy(dtype=float)
                        yerr  = g[err_col].to_numpy(dtype=float)
                        lower = np.maximum(yv - yerr, np.finfo(float).tiny)
                        upper = yv + yerr
                        ok    = np.isfinite(x) & np.isfinite(yv) & np.isfinite(yerr) & (yv > 0) & (upper > 0)
                        if np.any(ok):
                            ax.fill_between(x[ok], lower[ok], upper[ok],
                                            color=line.get_color(), alpha=0.18, linewidth=0)
                ax.set_yscale("log")
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel} (log)", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td  (log scale)", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_log_{subj}.png", dpi=120)
        plt.close(fig)

/tmp/ipykernel_3421907/2421587572.py:44: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale("log")


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT SIGNAL CONTRAST  ΔS = S(N_hi) − S(N_lo)  vs G, Ld, Lcf, lcf
# Per subject: rows = ROIs, columns = directions, colour = td
# "_raw"  uses the full model:  sqrt( (M0·(f1·S_mixed+f2·S_tort))² + RN² )
# "_corr" uses the clean model: M0·(f1·S_mixed+f2·S_tort)  (no Rician term)
# ══════════════════════════════════════════════════════════════════════════════

td_cmap    = plt.cm.viridis
N_hi, N_lo = N_LIST[-1], N_LIST[0]
G_plot     = np.linspace(0, float(data.G_fit.max()), 300)
PEAK_GAMMA = 267.5221900   # rad / (ms · mT), proton gyromagnetic ratio
D0_PLOT    = D0_FIXED

def _x_for_td(td_ms, xvar):
    if xvar == "G":
        return G_plot.copy()
    D0, gamma = float(D0_PLOT), float(PEAK_GAMMA)
    l_d = np.sqrt(D0 * float(td_ms))
    l_G = np.full_like(G_plot, np.nan)
    valid = G_plot > 0
    l_G[valid] = (D0 / (gamma * G_plot[valid])) ** (1.0 / 3.0)
    Ld  = l_d / l_G
    Lcf = 1.5 ** 0.25 / np.sqrt(Ld)
    lcf = Lcf * l_G * 1e6
    if xvar == "Ld":   return Ld
    if xvar == "Lcf_a": return Lcf
    if xvar == "lcf":  return lcf
    raise ValueError(xvar)

X_AXES = [
    ("G",    G_LABEL),
    ("Ld",   "Lᵈ (dimensionless)"),
    ("Lcf_a", "Lcf (dimensionless)"),
    ("lcf",  "lcf [µm]"),
]

# ── precompute contrasts ───────────────────────────────────────────────────────
raw_contrast_store  = {}   # full model: sqrt( (M0·(f1·S_mixed+f2·S_tort))² + RN² )
corr_contrast_store = {}   # clean model: M0·(f1·S_mixed+f2·S_tort)

for (subj, roi, direction), store in fit_store.items():
    for td in store["tds"]:
        raw_per_N  = {}
        corr_per_N = {}
        for N in N_LIST:
            fit = store["fits"][td].get(N)
            if fit is None:
                continue
            rn = fit.get("RN", 0.0)
            with np.errstate(over="ignore", invalid="ignore"):
                s1 = M_ogse_mixed_offset(td, G_plot, N, td / N, fit["tc1"], fit["alpha1"], 1, D0_FIXED, 0, 0)
                s2 = M_ogse_tort(td, G_plot, N, td / N, fit["alpha2"], 1, D0_FIXED)
                bimodal  = fit["f1"] * s1 + fit["f2"] * s2
                raw_per_N[N]  = np.sqrt(bimodal**2 + (rn / fit["M0"])**2) if rn != 0.0 else bimodal
                corr_per_N[N] = bimodal
        if N_hi in raw_per_N and N_lo in raw_per_N:
            raw_contrast_store[(subj, roi, direction, td)]  = raw_per_N[N_hi]  - raw_per_N[N_lo]
        if N_hi in corr_per_N and N_lo in corr_per_N:
            corr_contrast_store[(subj, roi, direction, td)] = corr_per_N[N_hi] - corr_per_N[N_lo]


def _plot_contrast(cstore, xvar, xlabel, subj, suffix, title_tag):
    rois_subj = sorted(set(k[1] for k in cstore if k[0] == subj))
    dirs_subj = [d for d in DIRECTIONS if any(k[2] == d for k in cstore if k[0] == subj)]
    n_rois, n_dirs = len(rois_subj), len(dirs_subj)
    if n_rois == 0 or n_dirs == 0:
        return
    all_tds_subj = sorted(set(k[3] for k in cstore if k[0] == subj))
    colors_leg   = [td_cmap(i / max(1, len(all_tds_subj) - 1)) for i in range(len(all_tds_subj))]
    leg_handles  = [plt.Line2D([0], [0], color=c, linewidth=1.5) for c in colors_leg]
    leg_labels   = [f"td={td:.0f} ms" for td in all_tds_subj]
    td_color_map = {td: c for td, c in zip(all_tds_subj, colors_leg)}
    fig, axes = plt.subplots(n_rois, n_dirs, figsize=(4 * n_dirs, 3 * n_rois), squeeze=False, sharey="row")
    for i_roi, roi in enumerate(rois_subj):
        for i_dir, direction in enumerate(dirs_subj):
            ax = axes[i_roi, i_dir]
            tds_here = sorted(k[3] for k in cstore
                              if k[0] == subj and k[1] == roi and k[2] == direction)
            if not tds_here:
                ax.set_visible(False)
                continue
            for td in tds_here:
                ax.plot(_x_for_td(td, xvar), cstore[(subj, roi, direction, td)],
                        color=td_color_map[td], linewidth=1.2)
            ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
            if i_roi == 0:
                ax.set_title(direction, fontsize=9)
            if i_dir == 0:
                ax.set_ylabel(f"{roi}\nΔS (N={N_hi}−N={N_lo})", fontsize=7)
            ax.set_xlabel(xlabel, fontsize=7)
            ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)
            ax.tick_params(labelsize=6)
            ax.tick_params(axis="y", labelleft=True)
    fig.suptitle(f"{subj} — contrast ΔS (N={N_hi}−N={N_lo}) vs {xvar}  [{title_tag}]", fontsize=10)
    fig.legend(leg_handles, leg_labels,
               loc="upper center", ncol=len(leg_labels), fontsize=7, frameon=False,
               bbox_to_anchor=(0.5, 1.0), bbox_transform=fig.transFigure)
    fig.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig(OUT_DIR / f"contrast_vs_{xvar}_{subj}_{suffix}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)


all_subjs = sorted(set(k[0] for k in raw_contrast_store))
for xvar, xlabel in X_AXES:
    for subj in all_subjs:
        _plot_contrast(raw_contrast_store,  xvar, xlabel, subj, "raw",  "fit w/ RN")
        _plot_contrast(corr_contrast_store, xvar, xlabel, subj, "corr", "clean model")

print("Per-subject contrast figures saved (_raw and _corr).")

Per-subject contrast figures saved (_raw and _corr).
